# Inference — Heart Disease Risk Prediction

This notebook demonstrates loading the packaged model pipeline and running inference exactly the way the FastAPI service does — both use the
same `HeartDiseaseModel` wrapper (`src/heart_disease/predict.py`), so there is only one code path from raw patient features to a prediction.

Run `python -m heart_disease.train` first if `models/best_model.joblib` doesn't exist yet.

In [ ]:
import pandas as pd

from heart_disease.data import TARGET
from heart_disease.predict import FEATURE_ORDER, HeartDiseaseModel

pd.set_option("display.max_columns", 20)

## 1. Load the packaged model

`HeartDiseaseModel()` loads `models/best_model.joblib` — a single `sklearn.Pipeline`
containing both the fitted preprocessing (scaler + one-hot encoder) and the classifier, so
no separate preprocessing step is needed here.

In [ ]:
model = HeartDiseaseModel()
print("expected feature order:", FEATURE_ORDER)

## 2. Single patient prediction

Same payload shape the FastAPI `/predict` endpoint accepts as JSON.

In [ ]:
sample_patient = {
    "age": 63,
    "trestbps": 145,
    "chol": 233,
    "thalach": 150,
    "oldpeak": 2.3,
    "sex": 1,
    "cp": 1,
    "fbs": 1,
    "restecg": 2,
    "exang": 0,
    "slope": 3,
    "ca": 0,
    "thal": 6,
}

result = model.predict_one(sample_patient)
print(result)

## 3. Missing-feature error handling

`predict_one` validates that every required feature is present before touching the model.

In [ ]:
incomplete_patient = dict(sample_patient)
del incomplete_patient["age"]

try:
    model.predict_one(incomplete_patient)
except ValueError as exc:
    print("raised as expected:", exc)

## 4. Batch inference on held-out data

Score several rows from the cleaned dataset at once and compare predictions to the actual
labels. `predict_one` is written for single-record API requests, so for a batch we call the
underlying pipeline directly (`model.pipeline`) — the exact same object either way.

In [ ]:
from heart_disease.data import get_clean_dataset

df = get_clean_dataset()
batch = df.drop(columns=[TARGET]).sample(10, random_state=42)
actual = df.loc[batch.index, TARGET]

predictions = model.pipeline.predict(batch[FEATURE_ORDER])
probabilities = model.pipeline.predict_proba(batch[FEATURE_ORDER])[:, 1]

results_df = batch.copy()
results_df["actual"] = actual
results_df["predicted"] = predictions
results_df["probability"] = probabilities.round(4)
results_df[["actual", "predicted", "probability"]]

## 5. Accuracy on this sample

In [ ]:
accuracy = (results_df["actual"] == results_df["predicted"]).mean()
print(f"accuracy on sampled batch: {accuracy:.2%}")

## Summary

- `HeartDiseaseModel` is the single inference entry point shared by this notebook, the
  FastAPI `/predict` endpoint (`api/main.py`), and the test suite (`tests/test_predict.py`,
  `tests/test_api.py`) — there is no duplicated prediction logic to drift out of sync.
- It fails fast with a clear `ValueError` on missing features rather than silently producing
  a wrong prediction.
- Because the loaded object is the full preprocessing + classifier `Pipeline`, batch scoring
  works directly on a raw-feature DataFrame with no manual scaling/encoding step.